## Seção 0 · Instalação e Configuração do Ambiente

Preparação do ambiente de execução: instalação de dependências, detecção automática Colab/local e configuração da API Key do Google Gemini.

**Compatível com Google Colab e execução local.**

In [21]:
# @title 0.1 · Instalar dependências
!python -m pip install -q pdfplumber langchain-text-splitters sentence-transformers faiss-cpu rank-bm25 google-genai python-dotenv tqdm pandas numpy
print('\u2705 Dependências instaladas.')


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✅ Dependências instaladas.


In [22]:
# @title 0.2 · Configurar ambiente (Colab / local)
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('\u26a0\ufe0f  Execu\u00e7\u00e3o local detectada.')

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/pln-rag-normas-estruturais')
else:
    cwd = Path('.').resolve()
    PROJECT_ROOT = cwd
    for _ in range(5):
        if (cwd / 'data').exists() and (cwd / 'src').exists():
            PROJECT_ROOT = cwd
            break
        if cwd.parent == cwd:
            break
        cwd = cwd.parent

NORMS_DIR = PROJECT_ROOT / 'data' / 'norms'
EVAL_DIR  = PROJECT_ROOT / 'data' / 'eval'
INDEX_DIR = PROJECT_ROOT / 'index'
INDEX_DIR.mkdir(parents=True, exist_ok=True)

pdfs = list(NORMS_DIR.glob('*.pdf'))
print(f'\U0001f4c2 Raiz: {PROJECT_ROOT}')
print(f'\U0001f4c4 PDFs encontrados: {len(pdfs)}')
for p in pdfs:
    print(f'   - {p.name} ({p.stat().st_size / 1024**2:.1f} MB)')

⚠️  Execução local detectada.
📂 Raiz: /Users/israelmagalhaes/Documents/development/pln-rag-normas-estruturais
📄 PDFs encontrados: 3
   - NBR6123_2023_PROPOSTA.pdf (9.0 MB)
   - NBR6118_2023.pdf (3.1 MB)
   - NBR6120_2019.pdf (0.7 MB)


In [23]:
# @title 0.3 · Configurar API Key do Google Gemini
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / '.env')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')

if not GEMINI_API_KEY:
    GEMINI_API_KEY = input('Cole sua GEMINI_API_KEY: ').strip()

if GEMINI_API_KEY:
    print('\u2705 API Key configurada.')
else:
    print('\u26a0\ufe0f  API Key n\u00e3o configurada. Se\u00e7\u00e3o 5 (RAG/Gemini) n\u00e3o funcionar\u00e1.')

✅ API Key configurada.


## Seção 1 · Ingestão dos PDFs

Extração de texto página a página com **pdfplumber**, preservando tabelas com marcadores `[TABELA]...[/TABELA]`.

| doc_id   | Título                                               | Edição              |
|----------|------------------------------------------------------|---------------------|
| NBR6118  | Projeto de estruturas de concreto — Procedimento      | 2023                |
| NBR6120  | Ações para o cálculo de estruturas de edificações       | 2019                |
| NBR6123  | Forças devidas ao vento em edificações                 | 2023 (Proposta)     |

In [24]:
# @title 1.1 · Funções de ingestão (inline)
import re
from typing import Any

import pdfplumber
from tqdm import tqdm

# Metadados fixos por documento
DOCUMENT_METADATA = {
    'NBR6118': {
        'doc_id': 'NBR6118',
        'titulo': 'Projeto de estruturas de concreto \u2014 Procedimento',
        'fonte': 'ABNT',
        'edicao': '2023',
        'filename': 'NBR6118_2023.pdf',
    },
    'NBR6120': {
        'doc_id': 'NBR6120',
        'titulo': 'A\u00e7\u00f5es para o c\u00e1lculo de estruturas de edifica\u00e7\u00f5es',
        'fonte': 'ABNT',
        'edicao': '2019',
        'filename': 'NBR6120_2019.pdf',
    },
    'NBR6123': {
        'doc_id': 'NBR6123',
        'titulo': 'For\u00e7as devidas ao vento em edifica\u00e7\u00f5es',
        'fonte': 'ABNT',
        'edicao': '2023 (Proposta de Revis\u00e3o)',
        'filename': 'NBR6123_2023_PROPOSTA.pdf',
    },
}


def extract_text_from_pdf(pdf_path):
    """Extrai texto p\u00e1gina a p\u00e1gina de um PDF usando pdfplumber.
    Tabelas s\u00e3o serializadas como TSV entre marcadores [TABELA]...[/TABELA]."""
    pdf_path = Path(pdf_path)
    pages = []

    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            raw_text = page.extract_text() or ''

            table_texts = []
            for table in page.extract_tables():
                rows = []
                for row in table:
                    cleaned = [str(cell).strip() if cell else '' for cell in row]
                    rows.append('\t'.join(cleaned))
                if rows:
                    table_texts.append('[TABELA]\n' + '\n'.join(rows) + '\n[/TABELA]')

            combined = raw_text
            if table_texts:
                combined = raw_text + '\n\n' + '\n\n'.join(table_texts)

            if combined.strip():
                pages.append({'page_num': i, 'text': combined.strip()})

    return pages


def load_document(doc_id, norms_dir):
    """Carrega texto completo e metadados de um documento normativo."""
    if doc_id not in DOCUMENT_METADATA:
        raise ValueError(f"doc_id '{doc_id}' desconhecido. Op\u00e7\u00f5es: {list(DOCUMENT_METADATA.keys())}")

    meta = DOCUMENT_METADATA[doc_id].copy()
    pdf_path = Path(norms_dir) / meta['filename']

    if not pdf_path.exists():
        raise FileNotFoundError(f'PDF n\u00e3o encontrado: {pdf_path}')

    print(f'[ingest\u00e3o] Carregando {doc_id} ({pdf_path.name})...')
    pages = extract_text_from_pdf(pdf_path)
    full_text = '\n\n'.join(p['text'] for p in pages)

    return {
        **meta,
        'pages': pages,
        'n_pages': len(pages),
        'full_text': full_text,
        'n_chars': len(full_text),
    }


def load_all_documents(norms_dir):
    """Carrega todos os documentos normativos registrados."""
    documents = []
    doc_ids = list(DOCUMENT_METADATA.keys())

    for doc_id in tqdm(doc_ids, desc='Ingest\u00e3o de normas'):
        try:
            doc = load_document(doc_id, norms_dir)
            documents.append(doc)
            print(f'  \u2713 {doc_id}: {doc["n_pages"]} p\u00e1ginas, {doc["n_chars"]:,} caracteres')
        except FileNotFoundError as e:
            print(f'  \u2717 {doc_id}: {e}')

    print(f'\n[ingest\u00e3o] {len(documents)}/{len(doc_ids)} documentos carregados.')
    return documents


print('\u2705 Fun\u00e7\u00f5es de ingest\u00e3o definidas.')

✅ Funções de ingestão definidas.


In [25]:
# @title 1.2 · Carregar documentos
# ⏱️ ~1-3 min dependendo do tamanho dos PDFs
documents = load_all_documents(norms_dir=str(NORMS_DIR))

print(f'\n\u2705 {len(documents)} documentos carregados.')
for doc in documents:
    status = (
        f"{doc['n_pages']} p\u00e1ginas, {doc['n_chars']:,} chars"
        if doc['n_chars'] > 0
        else '\u26a0\ufe0f 0 chars (PDF pode ter prote\u00e7\u00e3o)'
    )
    print(f"  {doc['doc_id']}: {status}")

Ingestão de normas:   0%|          | 0/3 [00:00<?, ?it/s]

[ingestão] Carregando NBR6118 (NBR6118_2023.pdf)...


Ingestão de normas:   0%|          | 0/3 [00:09<?, ?it/s]


KeyboardInterrupt: 

## Seção 2 · Chunking Hierárquico

Segmentação dos documentos em chunks rastreavéis com detecção automática de seção normativa.

**Parâmetros escolhidos:**
- `chunk_size = 800` caracteres — captura tabela típica (~400–600 chars) + parágrafo de contexto
- `chunk_overlap = 120` caracteres — ~1–2 linhas, preserva continuidade de enumerações
- Separadores: `["\n\n", "\n", ". ", " ", ""]` — prioriza quebras naturais

**Formato do chunk_id:** `NBR6120#3.2_0012` (doc_id + seção + sequencial)

In [26]:
# @title 2.1 · Funções de chunking (inline)
import re
import statistics
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120

# Regex para detectar seção normativa: "4.2.3 Ações variáveis"
_SECTION_RE = re.compile(
    r'(?:^|\n)\s*(\d{1,2}(?:\.\d{1,3}){0,4})\s+[A-Z\u00c0-\u017e]',
    re.MULTILINE,
)


def _detect_section(text):
    """Extrai o n\u00famero de se\u00e7\u00e3o normativa mais espec\u00edfico presente no texto."""
    matches = _SECTION_RE.findall(text)
    if matches:
        return matches[-1]
    return 'intro'


def _make_chunk_id(doc_id, section, seq):
    """Gera ID \u00fanico: NBR6120#3.2_0012"""
    return f'{doc_id}#{section}_{seq:04d}'


def chunk_document(doc, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """Segmenta um documento normativo em chunks hier\u00e1rquicos rastre\u00e1veis."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n\n', '\n', '. ', ' ', ''],
        length_function=len,
    )

    raw_chunks = splitter.split_text(doc['full_text'])

    chunks = []
    for seq, text in enumerate(raw_chunks, start=1):
        section = _detect_section(text)
        chunk_id = _make_chunk_id(doc['doc_id'], section, seq)

        chunks.append({
            'chunk_id': chunk_id,
            'doc_id': doc['doc_id'],
            'titulo': doc['titulo'],
            'fonte': doc['fonte'],
            'edicao': doc['edicao'],
            'secao': section,
            'texto': text.strip(),
            'n_chars': len(text.strip()),
        })

    return chunks


def chunk_documents(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """Segmenta uma lista de documentos normativos."""
    all_chunks = []

    for doc in docs:
        doc_chunks = chunk_document(doc, chunk_size, chunk_overlap)
        all_chunks.extend(doc_chunks)
        print(f'[chunker] {doc["doc_id"]}: {len(doc_chunks)} chunks '
              f'(size={chunk_size}, overlap={chunk_overlap})')

    print(f'\n[chunker] Total: {len(all_chunks)} chunks gerados.')
    return all_chunks


def print_chunks_stats(chunks):
    """Exibe estat\u00edsticas de distribui\u00e7\u00e3o de tamanho dos chunks."""
    sizes = [c['n_chars'] for c in chunks]
    docs = {}
    for c in chunks:
        docs.setdefault(c['doc_id'], 0)
        docs[c['doc_id']] += 1

    print(f'\n{"="*60}')
    print('  ESTAT\u00cdSTICAS DOS CHUNKS')
    print(f'{"="*60}')
    print(f'  Total de chunks : {len(chunks)}')
    print(f'  Tamanho m\u00e9dio   : {statistics.mean(sizes):.0f} chars')
    print(f'  Tamanho mediano : {statistics.median(sizes):.0f} chars')
    print(f'  M\u00ednimo          : {min(sizes)} chars')
    print(f'  M\u00e1ximo          : {max(sizes)} chars')
    print(f'\n  Por documento:')
    for doc_id, count in sorted(docs.items()):
        print(f'    {doc_id}: {count} chunks')
    print(f'{"="*60}')


def find_table_chunks(chunks):
    """Filtra chunks que cont\u00eam marcadores de tabela [TABELA]."""
    return [c for c in chunks if '[TABELA]' in c['texto']]


print('\u2705 Fun\u00e7\u00f5es de chunking definidas.')

✅ Funções de chunking definidas.


In [27]:
# @title 2.2 · Gerar chunks
all_chunks = chunk_documents(documents)
print(f'\n\u2705 Total: {len(all_chunks)} chunks gerados.')
print_chunks_stats(all_chunks)

[chunker] NBR6118: 5743 chunks (size=800, overlap=120)
[chunker] NBR6120: 303 chunks (size=800, overlap=120)
[chunker] NBR6123: 0 chunks (size=800, overlap=120)

[chunker] Total: 6046 chunks gerados.

✅ Total: 6046 chunks gerados.

  ESTATÍSTICAS DOS CHUNKS
  Total de chunks : 6046
  Tamanho médio   : 605 chars
  Tamanho mediano : 696 chars
  Mínimo          : 1 chars
  Máximo          : 799 chars

  Por documento:
    NBR6118: 5743 chunks
    NBR6120: 303 chunks


In [28]:
# @title 2.3 · Inspeção: chunks com tabelas
table_chunks = find_table_chunks(all_chunks)
print(f'Chunks com tabelas: {len(table_chunks)} / {len(all_chunks)}')

if table_chunks:
    print(f'\nExemplo de chunk com tabela:')
    print(f'  chunk_id: {table_chunks[0]["chunk_id"]}')
    print(f'  doc_id:   {table_chunks[0]["doc_id"]}')
    print(f'  se\u00e7\u00e3o:    {table_chunks[0]["secao"]}')
    print(f'  n_chars:  {table_chunks[0]["n_chars"]}')
    print(f'\n  Texto (primeiros 400 chars):')
    print(f'  {table_chunks[0]["texto"][:400]}')

Chunks com tabelas: 113 / 6046

Exemplo de chunk com tabela:
  chunk_id: NBR6118#intro_0722
  doc_id:   NBR6118
  seção:    intro
  n_chars:  683

  Texto (primeiros 400 chars):
  [TABELA]
(cid:38)(cid:79)(cid:68)(cid:86)(cid:86)(cid:72)(cid:3)(cid:71)(cid:72)(cid:3)
(cid:68)(cid:74)(cid:85)(cid:72)(cid:86)(cid:86)(cid:76)(cid:89)(cid:76)(cid:71)(cid:68)(cid:71)(cid:72)(cid:3)
(cid:68)(cid:80)(cid:69)(cid:76)(cid:72)(cid:81)(cid:87)(cid:68)(cid:79)	(cid:36)(cid:74)(cid:85)(cid:72)(cid:86)(cid:86)(cid:76)(cid:89)(cid:76)(cid:71)(cid:68)(cid:71)(cid:72)	(cid:38)(cid:79)(cid:6


## Seção 3 · Embeddings e Índice FAISS

**Modelo:** `neuralmind/bert-base-portuguese-cased` (BERT-PT-BR, ~400 MB).
Pré-treinado exclusivamente em português, excelente desempenho em similaridade semântica para textos técnicos.

**Índice:** `IndexFlatIP` (produto interno) com vetores L2-normalizados = busca por similaridade de cosseno.
Busca exata (é viável pois o corpus é pequeno: ~1.000–3.000 chunks).

In [29]:
# @title 3.1 · Carregar modelo de embedding
# ⏱️ Primeira execução: ~2-5 min (download ~400 MB)
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

EMBEDDING_MODEL = 'neuralmind/bert-base-portuguese-cased'
print(f'[indexer] Carregando modelo: {EMBEDDING_MODEL}')
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(f'\u2705 Modelo carregado. Dim: {embed_model.get_sentence_embedding_dimension()}')

[indexer] Carregando modelo: neuralmind/bert-base-portuguese-cased


No sentence-transformers model found with name neuralmind/bert-base-portuguese-cased. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23926.91it/s]
BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

✅ Modelo carregado. Dim: 768


In [ ]:
# @title 3.2 · Construir índice FAISS
# ⏱️ ~3-10 min dependendo do número de chunks e GPU
import time


def embed_chunks(chunks, model, batch_size=32):
    """Gera embeddings L2-normalizados para todos os chunks."""
    from tqdm import tqdm
    texts = [c['texto'] for c in chunks]
    print(f'[indexer] Gerando embeddings para {len(texts)} chunks...')
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return embeddings.astype(np.float32)


def build_faiss_index(chunks, model):
    """Constr\u00f3i \u00edndice FAISS IndexFlatIP a partir dos chunks."""
    embeddings = embed_chunks(chunks, model)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    print(f'[indexer] \u00cdndice criado: {index.ntotal} vetores (dim={dim})')
    return index, chunks


t0 = time.time()
faiss_index, indexed_chunks = build_faiss_index(all_chunks, embed_model)
print(f'\u2705 \u00cdndice constru\u00eddo em {time.time()-t0:.1f}s | {faiss_index.ntotal} vetores')

[indexer] Gerando embeddings para 6046 chunks...


Batches:   1%|          | 1/189 [00:10<34:07, 10.89s/it]


KeyboardInterrupt: 

In [30]:
# @title 3.3 · Persistir índice em disco)
import json, faiss
faiss_index = faiss.read_index(str(INDEX_DIR / 'faiss.index'))
with open(INDEX_DIR / 'chunks_metadata.json', 'r') as f:
    indexed_chunks = json.load(f)
print(f'✅ Índice carregado do disco: {faiss_index.ntotal} vetores')



def save_faiss_index(index, chunks, index_dir):
    """Salva \u00edndice FAISS, metadados dos chunks e info do modelo."""
    index_dir = Path(index_dir)
    index_dir.mkdir(parents=True, exist_ok=True)

    faiss.write_index(index, str(index_dir / 'faiss.index'))

    with open(index_dir / 'chunks_metadata.json', 'w', encoding='utf-8') as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    with open(index_dir / 'model_info.json', 'w', encoding='utf-8') as f:
        json.dump({'embedding_model': EMBEDDING_MODEL}, f)

    print(f'\u2705 \u00cdndice salvo em {index_dir}')
    for fname in ['faiss.index', 'chunks_metadata.json', 'model_info.json']:
        p = index_dir / fname
        if p.exists():
            print(f'   {fname} ({p.stat().st_size/1024:.1f} KB)')


save_faiss_index(faiss_index, indexed_chunks, index_dir=str(INDEX_DIR))

✅ Índice carregado do disco: 6046 vetores
✅ Índice salvo em /Users/israelmagalhaes/Documents/development/pln-rag-normas-estruturais/index
   faiss.index (18138.0 KB)
   chunks_metadata.json (5029.1 KB)
   model_info.json (0.1 KB)


## Seção 4 · Retrieval Semântico (FAISS)

Busca por similaridade de cosseno: a query é codificada pelo mesmo modelo de embedding e comparada contra todos os vetores do índice FAISS. Retorna os `k` chunks mais similares com score de relevância.

In [31]:
# @title 4.1 · Função de retrieval semântico


def retrieve_faiss(query, index, chunks, model, k=5):
    """Recupera os k chunks mais relevantes para uma query via FAISS."""
    query_vec = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    scores, indices = index.search(query_vec, k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk = chunks[idx].copy()
        chunk['rank'] = rank
        chunk['score'] = float(score)
        results.append(chunk)

    return results


def print_retrieval_results(results, query):
    """Exibe os resultados de retrieval de forma formatada."""
    print(f"\n{'='*70}")
    print(f'  Query: {query}')
    print(f"{'='*70}")
    for r in results:
        print(f"\n  #{r['rank']} | {r['chunk_id']} | score={r['score']:.4f}")
        print(f"  [{r['doc_id']}] Se\u00e7\u00e3o {r['secao']}")
        print(f"  {r['texto'][:200].replace(chr(10), ' ')}...")
    print(f"{'='*70}")


print('\u2705 Fun\u00e7\u00f5es de retrieval definidas.')

✅ Funções de retrieval definidas.


In [32]:
# @title 4.2 · Testar queries de exemplo
test_queries = [
    'Qual o valor t\u00edpico de carga acidental para um pavimento de escrit\u00f3rio?',
    'Quais fatores influenciam a velocidade de c\u00e1lculo do vento al\u00e9m da velocidade b\u00e1sica?',
    'Qual a diferen\u00e7a entre estados-limite \u00faltimos e estados-limite de servi\u00e7o?',
]

for query in test_queries:
    results = retrieve_faiss(query, faiss_index, indexed_chunks, embed_model, k=3)
    print_retrieval_results(results, query)


  Query: Qual o valor típico de carga acidental para um pavimento de escritório?

  #1 | NBR6120#intro_0293 | score=0.7391
  [NBR6120] Seção intro
  caminho de rolamento e qualquer uma ponte em outro caminho de rolamento não adjacente		Coeficiente de impacto vertical: 0 % para a dupla de pontes Forças horizontais: 50 % ambas as pontes, ou 100 % po...

  #2 | NBR6120#intro_0146 | score=0.7182
  [NBR6120] Seção intro
  E O g Para o teto da casa de máquinas de elevadores, verificar a necessidade de prever cargas concentradas D L variáveis para os ganchos de suspensão dos equipamentos (mínimo 40 kN por gancho). A E R ...

  #3 | NBR6120#intro_0153 | score=0.7124
  [NBR6120] Seção intro
  de paletes pesados (carga de utilização superior a 12 kN), deve ser realizado estudo específico. As cargas desta Norma não se aplicam ao projeto de porta-paletes e afins, que devem ser projetados conf...

  Query: Quais fatores influenciam a velocidade de cálculo do vento além da velocidade básica?

  #1 

In [33]:
# @title 4.3 · Comparar k=3, 5, 10 para a mesma query
compare_query = 'Quando \u00e9 permitido reduzir as cargas acidentais em um edif\u00edcio?'
print(f'Query: "{compare_query}"\n')

for k_val in [3, 5, 10]:
    results = retrieve_faiss(compare_query, faiss_index, indexed_chunks, embed_model, k=k_val)
    print(f'\n--- k={k_val} ---')
    for r in results:
        print(f'  #{r["rank"]} {r["chunk_id"]} (score={r["score"]:.4f}) [{r["doc_id"]}]')

Query: "Quando é permitido reduzir as cargas acidentais em um edifício?"


--- k=3 ---
  #1 NBR6120#intro_0166 (score=0.6753) [NBR6120]
  #2 NBR6120#intro_0208 (score=0.6708) [NBR6120]
  #3 NBR6120#intro_0146 (score=0.6668) [NBR6120]

--- k=5 ---
  #1 NBR6120#intro_0166 (score=0.6753) [NBR6120]
  #2 NBR6120#intro_0208 (score=0.6708) [NBR6120]
  #3 NBR6120#intro_0146 (score=0.6668) [NBR6120]
  #4 NBR6120#intro_0293 (score=0.6634) [NBR6120]
  #5 NBR6120#intro_0209 (score=0.6555) [NBR6120]

--- k=10 ---
  #1 NBR6120#intro_0166 (score=0.6753) [NBR6120]
  #2 NBR6120#intro_0208 (score=0.6708) [NBR6120]
  #3 NBR6120#intro_0146 (score=0.6668) [NBR6120]
  #4 NBR6120#intro_0293 (score=0.6634) [NBR6120]
  #5 NBR6120#intro_0209 (score=0.6555) [NBR6120]
  #6 NBR6120#intro_0145 (score=0.6540) [NBR6120]
  #7 NBR6120#intro_0218 (score=0.6534) [NBR6120]
  #8 NBR6120#intro_0220 (score=0.6531) [NBR6120]
  #9 NBR6120#intro_0287 (score=0.6515) [NBR6120]
  #10 NBR6120#intro_0038 (score=0.6498) [NBR6120]


## Seção 5 · Pipeline RAG (Google Gemini)

Pipeline completo: Retrieval (FAISS) + Generation (Gemini Flash 2.0).

**Estratégia de grounding:**
- O prompt instrui o LLM a responder EXCLUSIVAMENTE com base nos trechos normativos recuperados
- Citações no formato `[NBRxxxx#seção]` para rastreabilidade técnica
- Recusa educada para perguntas fora do corpus normativo

**Modos:**
- `baseline`: prompt direto com instrução de grounding e citações
- `improved`: chain-of-thought, verificação cruzada entre normas, formato estruturado

In [34]:
# @title 5.1 · Templates de prompt (inline)

SYSTEM_PROMPT_BASELINE = """\
Voc\u00ea \u00e9 um assistente t\u00e9cnico especializado em normas brasileiras de \
engenharia estrutural (ABNT).

REGRAS OBRIGAT\u00d3RIAS:
1. Responda APENAS com base nos trechos normativos fornecidos abaixo.
2. Cite as fontes ao longo da resposta usando o formato [NBRxxxx#se\u00e7\u00e3o], \
por exemplo: [NBR6118#13.2.4].
3. Se a informa\u00e7\u00e3o solicitada N\u00c3O estiver nos trechos fornecidos, responda \
exatamente: \"N\u00e3o encontrei informa\u00e7\u00e3o suficiente nas normas consultadas \
para responder esta pergunta.\"
4. N\u00c3O invente, extrapole ou use conhecimento externo \u00e0s normas.
5. Se a pergunta for sobre pre\u00e7os, or\u00e7amentos, marcas comerciais ou \
qualquer tema N\u00c3O normativo, recuse educadamente explicando que o sistema \
consulta apenas normas t\u00e9cnicas ABNT.
6. Preserve valores num\u00e9ricos, unidades e condi\u00e7\u00f5es exatamente como \
aparecem nos trechos.
"""

SYSTEM_PROMPT_IMPROVED = """\
Voc\u00ea \u00e9 um assistente t\u00e9cnico especializado em normas brasileiras de \
engenharia estrutural (ABNT).

INSTRU\u00c7\u00d5ES DE RACIOC\u00cdNIO:
Antes de responder, siga estas etapas mentalmente:
1. IDENTIFIQUE quais trechos normativos s\u00e3o relevantes para a pergunta.
2. VERIFIQUE se h\u00e1 informa\u00e7\u00e3o suficiente nos trechos para uma resposta \
fundamentada.
3. Se m\u00faltiplas normas abordam o tema, CRUZE as refer\u00eancias para uma \
resposta integrada.
4. CONFIRME que cada afirma\u00e7\u00e3o na sua resposta \u00e9 suportada por pelo menos \
um trecho fornecido.

REGRAS OBRIGAT\u00d3RIAS:
1. Responda APENAS com base nos trechos normativos fornecidos abaixo.
2. Cite as fontes ao longo da resposta usando o formato [NBRxxxx#se\u00e7\u00e3o], \
por exemplo: [NBR6118#13.2.4].
3. Se a informa\u00e7\u00e3o solicitada N\u00c3O estiver nos trechos fornecidos, responda \
exatamente: \"N\u00e3o encontrei informa\u00e7\u00e3o suficiente nas normas consultadas \
para responder esta pergunta.\"
4. N\u00c3O invente, extrapole ou use conhecimento externo \u00e0s normas.
5. Se a pergunta for sobre pre\u00e7os, or\u00e7amentos, marcas comerciais ou \
qualquer tema N\u00c3O normativo, recuse educadamente explicando que o sistema \
consulta apenas normas t\u00e9cnicas ABNT.
6. Preserve valores num\u00e9ricos, unidades e condi\u00e7\u00f5es exatamente como \
aparecem nos trechos.

FORMATO DE RESPOSTA:
- Comece com uma resposta objetiva e direta.
- Em seguida, detalhe com base nos trechos normativos, citando cada fonte.
- Se houver valores em tabelas, apresente-os de forma organizada.
- Finalize com a lista de refer\u00eancias normativas consultadas.
"""


def format_context(results):
    """Formata os chunks recuperados como bloco de contexto para o prompt."""
    parts = []
    for r in results:
        header = (
            f"[Fonte: {r['chunk_id']} | "
            f"{r['doc_id']} \u2014 Se\u00e7\u00e3o {r['secao']} | "
            f"Relev\u00e2ncia: {r['score']:.3f}]"
        )
        parts.append(f"{header}\n{r['texto']}")
    return '\n\n---\n\n'.join(parts)


def build_prompt(question, context, mode='baseline'):
    """Monta o prompt completo para envio ao LLM."""
    system = (
        SYSTEM_PROMPT_BASELINE if mode == 'baseline'
        else SYSTEM_PROMPT_IMPROVED
    )
    return (
        f'{system}\n'
        f'TRECHOS NORMATIVOS RECUPERADOS:\n'
        f'{context}\n\n'
        f'PERGUNTA DO USU\u00c1RIO:\n{question}\n\n'
        f'RESPOSTA:'
    )


print('\u2705 Templates de prompt e fun\u00e7\u00f5es auxiliares definidos.')

✅ Templates de prompt e funções auxiliares definidos.


In [35]:
# @title 5.2 · Função RAG query
# ⚠️ Rate-limited: ~5 req/min na API gratuita do Gemini Flash 2.5
# Se receber erro 429, aguarde 60s e tente novamente manualmente.
from google import genai
import time as _time


def rag_query(question, index, chunks, model, gemini_api_key,
              k=5, mode='baseline', gemini_model='gemini-2.0-flash'):
    """Pipeline RAG completo: retrieval + gera\u00e7\u00e3o com Gemini."""
    # Configure API
    llm_client = genai.Client(api_key=gemini_api_key)

    # Retrieval
    t0 = _time.time()
    results = retrieve_faiss(question, index, chunks, model, k=k)
    t_ret = _time.time() - t0

    # Prompt
    context = format_context(results)
    prompt = build_prompt(question, context, mode=mode)

    # Generation with retry
    t1 = _time.time()
    answer = ''
    for attempt, wait in enumerate([0, 10, 30, 60]):
        if attempt > 0:
            print(f'[rag] Rate limit. Aguardando {wait}s...')
            _time.sleep(wait)
        try:
            answer = llm_client.models.generate_content(model=gemini_model, contents=prompt).text
            break
        except Exception as e:
            if '429' not in str(e) or attempt == 3:
                answer = f'\u26a0\ufe0f Erro: {e}'
                break
    t_gen = _time.time() - t1

    return {
        'answer': answer,
        'sources': results,
        'mode': mode,
        'k': k,
        'latency': {
            'retrieval_s': round(t_ret, 3),
            'generation_s': round(t_gen, 3),
            'total_s': round(t_ret + t_gen, 3),
        },
    }


def display_rag_result(result, question):
    """Exibe resultado RAG de forma formatada."""
    print(f"\n{'='*70}")
    print(f'  Pergunta: {question}')
    print(f'  Modo: {result["mode"]} | k={result["k"]}')
    print(f'  Lat\u00eancia: retrieval={result["latency"]["retrieval_s"]:.3f}s, '
          f'gera\u00e7\u00e3o={result["latency"]["generation_s"]:.3f}s, '
          f'total={result["latency"]["total_s"]:.3f}s')
    print(f"{'='*70}")
    print(f'\nResposta:\n{result["answer"]}')
    print(f"\n{'---'*20}")
    print('Fontes recuperadas:')
    for s in result['sources']:
        print(f'  #{s["rank"]} {s["chunk_id"]} (score={s["score"]:.4f})')
    print(f"{'='*70}")


print('\u2705 Fun\u00e7\u00e3o rag_query definida.')

✅ Função rag_query definida.


In [36]:
# @title 5.3 · Teste RAG (exemplos)
# ⚠️ Rate-limited — teste manual recomendado
# Execute esta célula manualmente. Se receber erro 429, aguarde 60s.

if GEMINI_API_KEY:
    # Teste 1: factual_direta
    q1 = 'Qual o valor t\u00edpico de carga acidental para um pavimento de escrit\u00f3rio?'
    print('\n\U0001f9ea Teste 1: factual_direta')
    r1 = rag_query(q1, faiss_index, indexed_chunks, embed_model, GEMINI_API_KEY, k=5, mode='baseline')
    display_rag_result(r1, q1)

    print('\n\n')

    # Teste 2: fora_do_corpus (deve recusar)
    q2 = 'Como calcular o pre\u00e7o do m\u00b3 de concreto para uma obra em Bras\u00edlia?'
    print('\U0001f9ea Teste 2: fora_do_corpus (esperado: recusa educada)')
    r2 = rag_query(q2, faiss_index, indexed_chunks, embed_model, GEMINI_API_KEY, k=5, mode='baseline')
    display_rag_result(r2, q2)
else:
    print('\u26a0\ufe0f API Key n\u00e3o configurada. Pule esta c\u00e9lula ou configure na Se\u00e7\u00e3o 0.3.')


🧪 Teste 1: factual_direta

  Pergunta: Qual o valor típico de carga acidental para um pavimento de escritório?
  Modo: baseline | k=5
  Latência: retrieval=0.207s, geração=0.937s, total=1.144s

Resposta:
Não encontrei informação suficiente nas normas consultadas para responder esta pergunta.


------------------------------------------------------------
Fontes recuperadas:
  #1 NBR6120#intro_0293 (score=0.7391)
  #2 NBR6120#intro_0146 (score=0.7182)
  #3 NBR6120#intro_0153 (score=0.7124)
  #4 NBR6120#intro_0044 (score=0.7078)
  #5 NBR6120#intro_0220 (score=0.7020)



🧪 Teste 2: fora_do_corpus (esperado: recusa educada)

  Pergunta: Como calcular o preço do m³ de concreto para uma obra em Brasília?
  Modo: baseline | k=5
  Latência: retrieval=0.059s, geração=0.855s, total=0.915s

Resposta:
Não encontrei informação suficiente nas normas consultadas para responder esta pergunta.


------------------------------------------------------------
Fontes recuperadas:
  #1 NBR6120#intro_0150 (sc

## Seção 6 · Avaliação Baseline (Recall@k)

**Definição de Recall@k:**

Para cada pergunta com evidência esperada (não-nula):

```
hit(q, k) = 1  se algum chunk em top-k contém o doc_id da evidência
           = 0  caso contrário

Recall@k = (nº de hits) / (nº total de perguntas avaliadas)
```

**Correspondência conservadora por `doc_id`:** o golden_set usa referências como `"NBR6120#Tabela_X"` enquanto os chunk_ids têm formato `"NBR6120#3.2_0012"`. Verificamos apenas se o `doc_id` do chunk recuperado corresponde ao `doc_id` da evidência esperada.

**Perguntas `fora_do_corpus`** (evidencia_esperada = null) são excluídas do cálculo de Recall@k.

In [37]:
# @title 6.1 · Carregar golden_set e exibir distribuição
import json
import pandas as pd

golden_set_path = EVAL_DIR / 'golden_set.json'

with open(golden_set_path, encoding='utf-8') as f:
    golden_set = json.load(f)

print(f'[evaluator] Golden set carregado: {len(golden_set)} perguntas.')

# Distribuição por categoria
categorias = {}
for q in golden_set:
    cat = q['categoria']
    categorias[cat] = categorias.get(cat, 0) + 1

print('\nDistribui\u00e7\u00e3o por categoria:')
for cat, count in sorted(categorias.items()):
    print(f'  {cat}: {count} perguntas')

evaluable = [q for q in golden_set if q['evidencia_esperada'] is not None]
out_of_scope = [q for q in golden_set if q['evidencia_esperada'] is None]
print(f'\nAvali\u00e1veis (com evid\u00eancia): {len(evaluable)}')
print(f'Fora do corpus (exclu\u00eddas): {len(out_of_scope)}')

[evaluator] Golden set carregado: 21 perguntas.

Distribuição por categoria:
  factual_direta: 16 perguntas
  fora_do_corpus: 1 perguntas
  multi_trecho: 4 perguntas

Avaliáveis (com evidência): 20
Fora do corpus (excluídas): 1


In [38]:
# @title 6.2 · Funções de avaliação (inline)

EVAL_K_VALUES = [3, 5, 10]


def _extract_doc_id(evidencia):
    """Extrai doc_id de 'NBR6120#Tabela_X' -> 'NBR6120'."""
    return evidencia.split('#')[0]


def _is_hit(results, evidencias):
    """Verifica se algum resultado recuperado corresponde a alguma evid\u00eancia.
    Correspond\u00eancia baseada em doc_id (conservadora)."""
    retrieved_doc_ids = {r['doc_id'] for r in results}
    expected_doc_ids = {_extract_doc_id(e) for e in evidencias}
    return bool(retrieved_doc_ids & expected_doc_ids)


def run_evaluation(retrieve_fn, golden_set, k_values=EVAL_K_VALUES):
    """Executa avalia\u00e7\u00e3o Recall@k completa.

    Par\u00e2metros
    ----------
    retrieve_fn : callable
        Fun\u00e7\u00e3o com assinatura retrieve_fn(query, k) -> list[dict]
    golden_set : list[dict]
        Perguntas com evidencia_esperada.
    k_values : list[int]
        Valores de k para avalia\u00e7\u00e3o.

    Retorna
    -------
    dict com recall_at_k, n_questions, n_out_of_scope, details_df, summary_df
    """
    evaluable = [q for q in golden_set if q['evidencia_esperada'] is not None]
    out_of_scope = [q for q in golden_set if q['evidencia_esperada'] is None]

    print(f'[evaluator] Perguntas avali\u00e1veis: {len(evaluable)}')
    print(f'[evaluator] Perguntas fora do corpus (exclu\u00eddas): {len(out_of_scope)}')

    rows = []
    max_k = max(k_values)

    for q in tqdm(evaluable, desc='Avaliando retrieval'):
        # Normaliza evid\u00eancia para lista
        ev_raw = q['evidencia_esperada']
        evidencias = ev_raw if isinstance(ev_raw, list) else [ev_raw]

        # Recupera top-max_k uma \u00fanica vez por efici\u00eancia
        results = retrieve_fn(q['pergunta'], k=max_k)

        hits = {}
        for k in k_values:
            top_k_results = results[:k]
            hits[k] = _is_hit(top_k_results, evidencias)

        row = {
            'id': q['id'],
            'categoria': q['categoria'],
            'pergunta': q['pergunta'][:80] + '...',
            'evidencias': ', '.join(evidencias),
        }
        for k in k_values:
            row[f'hit@{k}'] = hits[k]
            row[f'retrieved_docs@{k}'] = ', '.join(
                {r['doc_id'] for r in results[:k]}
            )
        rows.append(row)

    details_df = pd.DataFrame(rows)

    # Calcula Recall@k
    recall_at_k = {}
    summary_rows = []

    for k in k_values:
        hits_total = details_df[f'hit@{k}'].sum()
        recall = hits_total / len(evaluable)
        recall_at_k[k] = recall
        summary_rows.append({
            'k': k,
            'hits': int(hits_total),
            'total': len(evaluable),
            'recall@k': round(recall, 4),
        })

    summary_df = pd.DataFrame(summary_rows)

    return {
        'recall_at_k': recall_at_k,
        'n_questions': len(evaluable),
        'n_out_of_scope': len(out_of_scope),
        'details_df': details_df,
        'summary_df': summary_df,
    }


print('\u2705 Fun\u00e7\u00f5es de avalia\u00e7\u00e3o definidas.')

✅ Funções de avaliação definidas.


In [39]:
# @title 6.3 · Executar avaliação baseline
# ⏱️ ~30s dependendo do hardware


def faiss_retrieve_fn(query, k):
    return retrieve_faiss(query, faiss_index, indexed_chunks, embed_model, k=k)


baseline_results = run_evaluation(faiss_retrieve_fn, golden_set)
print('\n\u2705 Avalia\u00e7\u00e3o baseline conclu\u00edda.')

[evaluator] Perguntas avaliáveis: 20
[evaluator] Perguntas fora do corpus (excluídas): 1


Avaliando retrieval: 100%|██████████| 20/20 [00:01<00:00, 10.79it/s]



✅ Avaliação baseline concluída.


In [40]:
# @title 6.4 · Relatório Recall@k
print(f"\n{'='*70}")
print('  RELAT\u00d3RIO DE AVALIA\u00c7\u00c3O \u2014 RECALL@K (RETRIEVER BASELINE FAISS)')
print(f"{'='*70}")
print(f"  Perguntas avaliadas    : {baseline_results['n_questions']}")
print(f"  Fora do corpus (excl.) : {baseline_results['n_out_of_scope']}")
print()

print(baseline_results['summary_df'].to_string(index=False))

print(f"\n{'='*70}")
print('  RECALL@K')
for k in [3, 5, 10]:
    r = baseline_results['recall_at_k'][k]
    bar = '\u2588' * int(r * 20)
    print(f'  Recall@{k:2d}: {r:.1%} {bar}')
print(f"{'='*70}")


  RELATÓRIO DE AVALIAÇÃO — RECALL@K (RETRIEVER BASELINE FAISS)
  Perguntas avaliadas    : 20
  Fora do corpus (excl.) : 1

 k  hits  total  recall@k
 3    10     20       0.5
 5    10     20       0.5
10    10     20       0.5

  RECALL@K
  Recall@ 3: 50.0% ██████████
  Recall@ 5: 50.0% ██████████
  Recall@10: 50.0% ██████████


In [41]:
# @title 6.5 · Detalhamento por pergunta + análise de misses
detail_cols = (
    ['id', 'categoria', 'evidencias']
    + [c for c in baseline_results['details_df'].columns if c.startswith('hit@')]
)
print('Detalhamento por pergunta:')
print(baseline_results['details_df'][detail_cols].to_string(index=False))

# Análise de misses no k=5
misses = baseline_results['details_df'][~baseline_results['details_df']['hit@5']]
if len(misses) > 0:
    print(f'\n\u26a0\ufe0f Perguntas com MISS no Recall@5 ({len(misses)}):')
    for _, row in misses.iterrows():
        print(f"  id={row['id']} | {row['categoria']} | evid\u00eancia={row['evidencias']}")
        print(f"    Pergunta: {row['pergunta']}")
        print(f"    Docs recuperados @5: {row['retrieved_docs@5']}")
else:
    print('\n\u2705 Nenhum MISS no Recall@5.')

Detalhamento por pergunta:
 id      categoria                                    evidencias  hit@3  hit@5  hit@10
  1 factual_direta                              NBR6120#Tabela_X   True   True    True
  2 factual_direta                              NBR6120#Tabela_Y   True   True    True
  3 factual_direta                        NBR6120#Reducao_Cargas   True   True    True
  4 factual_direta                     NBR6120#Acoes_Permanentes   True   True    True
  5 factual_direta                NBR6120#Metodologia_Estimativa   True   True    True
  6 factual_direta                            NBR6120#Coberturas   True   True    True
  7 factual_direta             NBR6120#Coberturas_Nao_Acessiveis   True   True    True
  8 factual_direta                     NBR6120#Divisorias_Moveis   True   True    True
  9   multi_trecho   NBR6120#Definicoes, NBR6118#Dimensionamento   True   True    True
 10 factual_direta                             NBR6123#Isopletas  False  False   False
 11 factual_dire

## Seção 7 · Melhoria: Hybrid Search (BM25 + FAISS / RRF)

Combinação de dois retrievers para melhorar recall:

1. **BM25 (léxico):** busca por correspondência exata de termos (bom para valores, siglas, números de seção)
2. **FAISS (semântico):** busca por similaridade de significado (bom para paráfrases e conceitos)

**Fusão via Reciprocal Rank Fusion (RRF):**
```
RRF_score(chunk) = Σ 1/(60 + rank_i)
```
onde `rank_i` é a posição do chunk no ranking de cada retriever. Chunks que aparecem bem ranqueados em ambos os retrievers recebem score mais alto.

In [42]:
# @title 7.1 · BM25 + HybridRetriever (inline)
import re
from rank_bm25 import BM25Okapi

_RRF_K = 60


def _tokenize(text):
    """Tokeniza texto: min\u00fasculas, divide em tokens alfanum\u00e9ricos."""
    return [tok for tok in re.split(r'\W+', text.lower()) if tok]


class HybridRetriever:
    """Retriever h\u00edbrido: BM25 (l\u00e9xico) + FAISS (sem\u00e2ntico) via RRF."""

    def __init__(self, chunks, faiss_index, embedding_model):
        self.chunks = chunks
        self.faiss_index = faiss_index
        self.embedding_model = embedding_model
        # Constr\u00f3i \u00edndice BM25
        tokenized_corpus = [_tokenize(chunk['texto']) for chunk in chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)

    def retrieve(self, query, k=5):
        """Retorna top-k chunks usando busca h\u00edbrida BM25+FAISS com RRF."""
        n = min(k * 3, len(self.chunks))

        # BM25 ranking
        bm25_scores = self.bm25.get_scores(_tokenize(query))
        bm25_top = np.argsort(bm25_scores)[::-1][:n]

        # FAISS ranking
        qvec = self.embedding_model.encode(
            [query], convert_to_numpy=True, normalize_embeddings=True
        ).astype(np.float32)
        _, faiss_top_raw = self.faiss_index.search(qvec, n)
        faiss_top = faiss_top_raw[0]

        # RRF fusion
        rrf = {}
        for rank, idx in enumerate(bm25_top, 1):
            rrf[int(idx)] = rrf.get(int(idx), 0.0) + 1.0 / (_RRF_K + rank)
        for rank, idx in enumerate(faiss_top, 1):
            rrf[int(idx)] = rrf.get(int(idx), 0.0) + 1.0 / (_RRF_K + rank)

        # Sort + top k
        top_k = sorted(rrf, key=lambda i: rrf[i], reverse=True)[:k]

        results = []
        for rank, idx in enumerate(top_k, 1):
            chunk = self.chunks[idx].copy()
            chunk['rank'] = rank
            chunk['score'] = rrf[idx]
            results.append(chunk)

        return results


print('\u2705 HybridRetriever definido.')

✅ HybridRetriever definido.


In [43]:
# @title 7.2 · Criar retriever híbrido e comparar com FAISS-only
hybrid_retriever = HybridRetriever(indexed_chunks, faiss_index, embed_model)
print('\u2705 HybridRetriever criado.')

# Comparação para a mesma query
test_query = 'Quando \u00e9 permitido reduzir as cargas acidentais em um edif\u00edcio?'
print(f'\nQuery: "{test_query}"\n')

faiss_results = retrieve_faiss(test_query, faiss_index, indexed_chunks, embed_model, k=5)
hybrid_results = hybrid_retriever.retrieve(test_query, k=5)

print('FAISS-only top-5:')
for r in faiss_results:
    print(f'  #{r["rank"]} {r["chunk_id"]} (score={r["score"]:.4f})')

print('\nHybrid (BM25+FAISS/RRF) top-5:')
for r in hybrid_results:
    print(f'  #{r["rank"]} {r["chunk_id"]} (rrf_score={r["score"]:.5f})')

✅ HybridRetriever criado.

Query: "Quando é permitido reduzir as cargas acidentais em um edifício?"

FAISS-only top-5:
  #1 NBR6120#intro_0166 (score=0.6753)
  #2 NBR6120#intro_0208 (score=0.6708)
  #3 NBR6120#intro_0146 (score=0.6668)
  #4 NBR6120#intro_0293 (score=0.6634)
  #5 NBR6120#intro_0209 (score=0.6555)

Hybrid (BM25+FAISS/RRF) top-5:
  #1 NBR6120#intro_0166 (rrf_score=0.03028)
  #2 NBR6120#intro_0278 (rrf_score=0.02973)
  #3 NBR6120#intro_0220 (rrf_score=0.02941)
  #4 NBR6120#intro_0282 (rrf_score=0.01613)
  #5 NBR6120#intro_0208 (rrf_score=0.01613)


## Seção 8 · Avaliação Pós-Melhoria e Comparação

Comparação direta entre:
- **Baseline:** FAISS-only (semântico puro)
- **Hybrid:** BM25 + FAISS com fusão RRF

Métrica: Recall@k para k = 3, 5, 10.

In [44]:
# @title 8.1 · Executar avaliação híbrida
# ⏱️ ~30s dependendo do hardware


def hybrid_retrieve_fn(query, k):
    return hybrid_retriever.retrieve(query, k=k)


hybrid_results_eval = run_evaluation(hybrid_retrieve_fn, golden_set)
print('\u2705 Avalia\u00e7\u00e3o h\u00edbrida conclu\u00edda.')

[evaluator] Perguntas avaliáveis: 20
[evaluator] Perguntas fora do corpus (excluídas): 1


Avaliando retrieval: 100%|██████████| 20/20 [00:02<00:00,  9.74it/s]

✅ Avaliação híbrida concluída.


In [46]:
# @title 8.2 · Tabela comparativa Recall@k
import pandas as pd

print('\n\U0001f4ca COMPARA\u00c7\u00c3O RECALL@K \u2014 Baseline FAISS vs Hybrid BM25+FAISS')
print('=' * 55)
print(f'{"k":>5} | {"Baseline FAISS":>15} | {"Hybrid BM25+FAISS":>18} | {"u0394":>6}')
print('-' * 55)
for k in [3, 5, 10]:
    b = baseline_results['recall_at_k'][k]
    h = hybrid_results_eval['recall_at_k'][k]
    delta = h - b
    sign = '+' if delta >= 0 else ''
    print(f'{k:>5} | {b:>14.1%} | {h:>17.1%} | {sign}{delta:.1%}')
print('=' * 55)


📊 COMPARAÇÃO RECALL@K — Baseline FAISS vs Hybrid BM25+FAISS
    k |  Baseline FAISS |  Hybrid BM25+FAISS |  u0394
-------------------------------------------------------
    3 |          50.0% |             45.0% | -5.0%
    5 |          50.0% |             45.0% | -5.0%
   10 |          50.0% |             45.0% | -5.0%


In [47]:
# @title 8.3 · Comparação detalhada por pergunta
# Merge dos DataFrames de detalhes
baseline_detail = baseline_results['details_df'][['id', 'categoria', 'pergunta', 'evidencias', 'hit@3', 'hit@5', 'hit@10']].copy()
hybrid_detail = hybrid_results_eval['details_df'][['id', 'hit@3', 'hit@5', 'hit@10']].copy()

baseline_detail = baseline_detail.rename(columns={'hit@3': 'base@3', 'hit@5': 'base@5', 'hit@10': 'base@10'})
hybrid_detail = hybrid_detail.rename(columns={'hit@3': 'hyb@3', 'hit@5': 'hyb@5', 'hit@10': 'hyb@10'})

comparison = baseline_detail.merge(hybrid_detail, on='id')

print('Compara\u00e7\u00e3o por pergunta (hit = True/False):')
print(comparison[['id', 'categoria', 'evidencias', 'base@5', 'hyb@5']].to_string(index=False))

# Análise de melhorias e pioras no k=5
improved = comparison[(~comparison['base@5']) & (comparison['hyb@5'])]
worsened = comparison[(comparison['base@5']) & (~comparison['hyb@5'])]

if len(improved) > 0:
    print(f'\n\u2705 Perguntas que MELHORARAM com hybrid (k=5): {len(improved)}')
    for _, row in improved.iterrows():
        print(f"  id={row['id']}: {row['pergunta']}")

if len(worsened) > 0:
    print(f'\n\u26a0\ufe0f Perguntas que PIORARAM com hybrid (k=5): {len(worsened)}')
    for _, row in worsened.iterrows():
        print(f"  id={row['id']}: {row['pergunta']}")

if len(improved) == 0 and len(worsened) == 0:
    print('\n\u2139\ufe0f Nenhuma mudan\u00e7a entre baseline e hybrid no k=5.')

Comparação por pergunta (hit = True/False):
 id      categoria                                    evidencias  base@5  hyb@5
  1 factual_direta                              NBR6120#Tabela_X    True   True
  2 factual_direta                              NBR6120#Tabela_Y    True   True
  3 factual_direta                        NBR6120#Reducao_Cargas    True   True
  4 factual_direta                     NBR6120#Acoes_Permanentes    True   True
  5 factual_direta                NBR6120#Metodologia_Estimativa    True   True
  6 factual_direta                            NBR6120#Coberturas    True   True
  7 factual_direta             NBR6120#Coberturas_Nao_Acessiveis    True   True
  8 factual_direta                     NBR6120#Divisorias_Moveis    True   True
  9   multi_trecho   NBR6120#Definicoes, NBR6118#Dimensionamento    True   True
 10 factual_direta                             NBR6123#Isopletas   False  False
 11 factual_direta                      NBR6123#Fatores_S1_S2_S3   False  Fa

In [48]:
# @title 8.4 · Resumo final
print('\n\u2705 Notebook conclu\u00eddo!')
print('   Ingest\u00e3o \u2192 Chunking \u2192 FAISS \u2192 Retrieval \u2192 RAG \u2192 Avalia\u00e7\u00e3o \u2192 Hybrid \u2192 Compara\u00e7\u00e3o')
print(f'\n\U0001f4ca Resultado final:')
for k in [3, 5, 10]:
    b = baseline_results['recall_at_k'][k]
    h = hybrid_results_eval['recall_at_k'][k]
    bar_b = '\u2591' * int(b * 20)
    bar_h = '\u2588' * int(h * 20)
    print(f'  Recall@{k:2d}: Baseline={b:.1%} {bar_b}')
    print(f'           Hybrid  ={h:.1%} {bar_h}')
print('\n\U0001f51c Pr\u00f3ximo: streamlit run app.py')


✅ Notebook concluído!
   Ingestão → Chunking → FAISS → Retrieval → RAG → Avaliação → Hybrid → Comparação

📊 Resultado final:
  Recall@ 3: Baseline=50.0% ░░░░░░░░░░
           Hybrid  =45.0% █████████
  Recall@ 5: Baseline=50.0% ░░░░░░░░░░
           Hybrid  =45.0% █████████
  Recall@10: Baseline=50.0% ░░░░░░░░░░
           Hybrid  =45.0% █████████

🔜 Próximo: streamlit run app.py
